# Lesson 02 - مائیکروسافٹ ایجنٹ فریم ورک کی کھوج

**مائیکروسافٹ ایجنٹ فریم ورک (MAF)** ایک متحدہ فریم ورک ہے جو AI ایجنٹس بنانے کے لیے ہے۔ یہ ایک صاف، ترکیب پذیر فن تعمیر فراہم کرتا ہے جس میں چار بنیادی جزو شامل ہیں:

- **کلائنٹ** – AI ماڈل کے اینڈ پوائنٹ سے جڑتا ہے اور مواصلات کو سنبھالتا ہے
- **ایجنٹ** – ایک کلائنٹ کو ہدایات اور ٹول کی تعریفوں کے ساتھ لپیٹتا ہے
- **ٹولز** – ایجنٹ کی صلاحیتوں کو کسٹم فنکشنز کے ساتھ بڑھاتے ہیں جنہیں ماڈل کال کر سکتا ہے
- **سیشن** – ملٹی ٹرن بات چیت کے لیے مکالمے کی تاریخ کو برقرار رکھتا ہے

اس سبق میں، ہم ایک **ٹریول بکنگ ایجنٹ** بنائیں گے جو ان تصورات کا استعمال کرتے ہوئے منزل کی دستیابی چیک کرتا ہے۔


## سیٹ اپ


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## ایجنٹ فریم ورک آرکیٹیکچر کو سمجھنا

مائیکروسافٹ ایجنٹ فریم ورک ایک تہہ دار آرکیٹیکچر کی پیروی کرتا ہے:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **کلائنٹ** – ایک `AzureAIProjectAgentProvider` Azure OpenAI ڈپلائمنٹ سے جڑتا ہے۔ یہ تصدیق، درخواست کی فارمیٹنگ، اور جواب کی تجزیہ کاری کو سنبھالتا ہے۔
2. **ایجنٹ** – کلائنٹ سے `provider.create_agent()` کے ذریعے بنایا جاتا ہے، ایجنٹ ماڈل کی رسائی کو ہدایات (سسٹم پرامپٹ) اور ٹولز کے ساتھ ملاتا ہے۔
3. **ٹولز** – Python فنکشنز جو `@tool` سے سجاۓ گئے ہیں جنہیں ایجنٹ کارروائی کرنے یا ڈیٹا حاصل کرنے کے لیے بلا سکتا ہے۔
4. **سیشن** – ایک `AgentSession` آبجیکٹ (جو `agent.create_session()` کے ذریعے بنایا جاتا ہے) جو گفتگو کی تاریخ ذخیرہ کرتا ہے، جس سے ملٹی ٹرن مکالمہ ممکن ہوتا ہے جہاں ایجنٹ پچھلا سیاق و سباق یاد رکھتا ہے۔

آئیے ہر تہہ کو قدم بہ قدم بنائیں۔


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool ڈیکوریٹر کے ساتھ ٹولز شامل کرنا

ٹولز ایجنٹس کو متن تیار کرنے سے آگے اقدامات کرنے دیتے ہیں۔ `@tool` ڈیکوریٹر ایک عام پائتھن فنکشن کو ایسی چیز میں تبدیل کر دیتا ہے جسے ایجنٹ کال کر سکتا ہے۔

اہم نکات:
- ماڈل کو ہر پیرامیٹر سمجھانے کے لیے `Annotated[type, "description"]` استعمال کریں۔
- ڈاک اسٹリング ٹول کی تفصیل بن جاتی ہے جو ماڈل دیکھتا ہے۔
- `approval_mode="never_require"` کا مطلب ہے کہ ٹول بغیر صارف کی تصدیق کے خودکار طور پر چلتا ہے۔


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## آلات کے ساتھ ایک ایجنٹ بنانا

اب ہم کلائنٹ، ہدایات، اور آلات کو ایک ایجنٹ میں مجتمع کرتے ہیں۔ `instructions` نظام کی ہدایت کے طور پر کام کرتے ہیں — یہ ایجنٹ کی شخصیت اور رویے کو متعین کرتے ہیں۔


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## متعدد مراسلات کے ساتھ سیشنز

ایک `AgentSession` (جو `agent.create_session()` کے ذریعے بنایا جاتا ہے) گفتگو میں تمام پیغامات کا پتہ رکھتا ہے۔ ایک ہی سیشن کو ہر `agent.run()` کال میں بھیج کر، ایجنٹ کو پوری گفتگو کی تاریخ تک رسائی حاصل ہوتی ہے اور وہ پچھلے پیغامات کی طرف رجوع کر سکتا ہے۔

ہم `tools=[check_destination_availability]` پاس کرتے ہیں تاکہ ایجنٹ ہر مرحلے پر ہماری دستیابی چیکر کو کال کر سکے۔


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## خلاصہ

اس سبق میں آپ نے مائیکروسافٹ ایجنٹ فریم ورک کے چار ستونوں کا جائزہ لیا:

| تصور | آپ نے کیا سیکھا |
|---------|------------------|
| **کلائنٹ** | `AzureAIProjectAgentProvider` اسناد کی بنیاد پر تصدیق کے ساتھ Azure OpenAI سے جڑتا ہے |
| **ایجنٹ** | `provider.create_agent()` ایک ماڈل کنکشن کو ہدایات اور نام کے ساتھ مربوط کرتا ہے |
| **ٹولز** | `@tool` ڈیکوریٹر ایجنٹ کے کال کرنے کے لیے پائتھن فنکشنز کو ظاہر کرتا ہے |
| **سیشن** | `agent.create_session()` متعدد مراحل کے دوران گفتگو کی تاریخ کو برقرار رکھتا ہے |

یہ بنیادی اجزاء مل کر ایسے ایجنٹس بناتے ہیں جو قدرتی گفتگو کر سکتے ہیں، بیرونی افعال کو کال کر سکتے ہیں، اور سیاق و سباق کو برقرار رکھتے ہیں — جو اگلے اسباق میں مزید پیش رفت ایجنٹک پیٹرنز کی بنیاد ہے۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈسکلیمر**:  
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ اگرچہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم یاد رکھیں کہ خودکار تراجم میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنی مادری زبان میں ہی معتبر ماخذ سمجھی جائے۔ اہم معلومات کے لیے پیشہ ور انسانی ترجمہ کرنے کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والے کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم پر نہیں ہوگی۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
